# Импорт библиотек и настройка среды

In [76]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score, average_precision_score

In [55]:
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_colwidth', None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Вспомогательные функции

## Функция создания labels для cut разделения

In [56]:
def make_labels(
    segments: list[float]
) -> list[str]:
    labels = []
    
    for i in range(len(segments) - 1):
        left = segments[i]
        right = segments[i + 1]

        if i == 0:
            labels.append(f"[{left}, {right}]")
        else:
            labels.append(f"({left}, {right}]")

    return labels

## Функция создания новых столбцов признака

In [57]:
def make_segments_cols(
    data: pd.DataFrame,
    feature: str,
    segments: list[float],
    special_values: list[int] = []
) -> pd.DataFrame:
    df = data.copy(deep=True)

    labels = make_labels(segments)

    segment_col_name = f"{feature}_segments"

    df[segment_col_name] = pd.cut(
        x=df[feature].dropna(),
        labels=labels,
        bins=segments,
        right=True,
        include_lowest=True
    )

    special_labels = []
    if special_values:
        for special_value in special_values:
            label = str(special_value)

            special_labels.append(label)

        new_categories = [
            label
            for label in special_labels
            if label not in df[segment_col_name].cat.categories
        ]

        df[segment_col_name] = (
            df[segment_col_name]
            .cat.add_categories(new_categories)
        )

        for special_value in special_values:
            mask = df[feature].eq(special_value)
            label = str(special_value)

            df.loc[mask, segment_col_name] = label

    cols = pd.get_dummies(
        df[segment_col_name],
        prefix=f"{feature}"
    )

    return cols
    

## Функция подготовки датасета

In [58]:
def prepare_dataset(
    data: pd.DataFrame,
    features: list[str],
    segmentss: list[list[float]],
    special_valuess: list[list[float]]
) -> pd.DataFrame:
    df = data.copy(deep=True)

    for feature, segments, special_values in zip(
        features,
        segmentss,
        special_valuess
    ):
        cols = make_segments_cols(
            data=df,
            feature=feature,
            segments=segments,
            special_values=special_values
        )

        df = df.drop(columns=feature)
        df = pd.concat([df, cols], axis=1)

    return df

## Функция оценки модели

In [108]:
def evaluate_model(
    model,
    X_test: pd.DataFrame,
    y_test: pd.Series
) -> tuple[float, float, float, float, float]:
    y_predict = model.predict(X_test)
    y_predict_score = model.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, y_predict)
    recall = recall_score(y_test, y_predict)
    roc_auc = roc_auc_score(y_test, y_predict_score)
    pr_auc = average_precision_score(y_test, y_predict_score)
    gini = 2 * roc_auc - 1

    return (precision, recall, roc_auc, pr_auc, gini)

# Загрузка датасета

In [59]:
data = pd.read_csv("../data/interim/borrowers.csv")

# Цель

Цель данного ноутбука - научиться оценивать надёжность заёмщиков, чтобы как можно полно отсеять плохих заёмщиков и сохранить хороших.

Чтобы этого достичь, нужно решить следующие задачи:
1. Подготовить данные для моделей;
2. Подобрать гиперпараметры для моделей;
3. Обучить модели;
4. Оценить качество полученных моделей;
5. Проинтерпретировать веса логистической регрессии.

# Методология

**1. Подготовка данных:**

Так как будут использоваться линейные и древовидные модели, то имеет смысл готовить отдельный тип датасета под каждый тип модели. То есть для линейных моделей будет усместна дискретизация признаков на несколько новых (соответствующих сегментам значений признака), а для градиентного бустинга от CatBoost ничего принимать не стоит.

Таким образом, для линейных моделей выполним дискретизацию, а для градиентного бустинга оставим исходный датасет.

**2. Подбор гиперпараметров:**

При помощи optuna подберём гиперпараметры для логистической регрессии и CatBoostClassifier. Прежде всего стоит обращать внимание на метрику ROC-AUC  так как она лучше учитывает дисбаланс классов.

**3. Обучение моделей:**

На подобранных гиперпараметрах, нужно произвести обучение моделей.

**4. Оценка качества моделей:**

На оставленной hold-out выборке произвести оценку качества моделей, прежде всего при помощи метрик ROC-AUC, PR-AUC, Gini, precision и recall нужно оценить качество моделей, а также сравнить модели между собой и принять решение о том стоит ли выбирать более сложную модель в угоду качеству (если она, конечно, отличается по качеству в положительную сторону).

**5. Интерпретация весов логистической регрессии:**

Для валидации модели стоит проинтерпретировать веса логистической регресии, чтобы понять насколько молель адекватно откликается на обнаруженные при анализе сегменты и маркеры риска.

# Подготовка признаков

Сегменты признаков и их специальные значения.

In [60]:
features = [
    "revolving_utilization",
    "age",
    "num_30_59_days_late",
    "debt_ratio",
    "monthly_income",
    "num_open_credit_lines",
    "num_90_days_late",
    "num_real_estate_loans",
    "num_60_89_days_late",
    "num_dependents"
]

segmentss = [
    [0.0, 0.2, 0.4, 0.5, 0.7, 1.0, 2.0, 5.0, np.inf],
    [0.0, 22.0, 30.0, 45.0, 55.0, 75.0, np.inf],
    [1, 2, np.inf],
    [0.0, 0.1, 0.7, 1.0, 4.0, np.inf],
    [0.0, 900.0, 3000.0, 4500.0, 6500.0, 13000.0, np.inf],
    [1, 2, np.inf],
    [1, 2, np.inf],
    [1, 3, np.inf],
    [1, 2, np.inf],
    [1, 2, np.inf]
]

special_valuess = [
    [],
    [],
    [0, 96, 98],
    [0],
    [],
    [0],
    [0, 96, 98],
    [0],
    [0, 96, 98],
    [0]
]

Подготовка признаков.

In [61]:
transformed_data = prepare_dataset(
    data=data,
    features=features,
    segmentss=segmentss,
    special_valuess=special_valuess
)

Разделение данных на train и test.

In [62]:
X = data.drop(columns="target").fillna(-1)
y = data["target"]

X_transformed = transformed_data.drop(columns="target")
y_transformed = transformed_data["target"]

In [63]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    shuffle=True,
    random_state=42,
    stratify=y
)

X_t_train, X_t_test, y_t_train, y_t_test = train_test_split(
    X_transformed,
    y_transformed,
    test_size=0.3,
    shuffle=True,
    random_state=42,
    stratify=y_transformed
)

# Подбор гиперпараметров

## Подбор гиперпараметров для логистической регрессии

Функция вычисления оптимальных гиперпараметров.

In [64]:
def objective(trial):
    C = trial.suggest_float(
        "C",
        1e-3,
        10,
        log=True
    )

    class_weight = trial.suggest_categorical(
        "class_weight",
        [None, "balanced"]
    )

    model = LogisticRegression(
        solver="lbfgs",
        penalty="l2",
        C=C,
        class_weight=class_weight,
        max_iter=1000,
        random_state=42
    )

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    score = cross_val_score(
        model,
        X_t_train,
        y_t_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    ).mean()

    return score

Подбор гиперпараметров.

In [65]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=20
)

[I 2026-08-20 03:19:57,227] A new study created in memory with name: no-name-926cdcfb-9b87-476c-97f4-7a3a603262a9
[I 2026-08-20 03:20:00,523] Trial 0 finished with value: 0.8566492488338169 and parameters: {'C': 0.5320915322986896, 'class_weight': None}. Best is trial 0 with value: 0.8566492488338169.
[I 2026-08-20 03:20:03,477] Trial 1 finished with value: 0.8578014274530394 and parameters: {'C': 5.367977004351217, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.8578014274530394.
[I 2026-08-20 03:20:05,557] Trial 2 finished with value: 0.8566351000384722 and parameters: {'C': 0.13927394201160875, 'class_weight': None}. Best is trial 1 with value: 0.8578014274530394.
[I 2026-08-20 03:20:06,355] Trial 3 finished with value: 0.8564837226605582 and parameters: {'C': 0.011864865347399379, 'class_weight': None}. Best is trial 1 with value: 0.8578014274530394.
[I 2026-08-20 03:20:07,924] Trial 4 finished with value: 0.8576679973911873 and parameters: {'C': 0.04661547218979179, 'cl

Сохранение оптимальных гиперпараметров.

In [66]:
logistic_regression_best_params = study.best_params

## Подбор параметров для CatBoostClassifier

Функция вычисления оптимальных гиперпараметров.

In [67]:
def objective(trial):
    model = CatBoostClassifier(
        iterations=trial.suggest_int(
            "iterations",
            300,
            700,
            step=100
        ),

        learning_rate=trial.suggest_float(
            "learning_rate",
            0.02,
            0.08,
            log=True
        ),

        depth=trial.suggest_int(
            "depth",
            4,
            7
        ),

        l2_leaf_reg=trial.suggest_float(
            "l2_leaf_reg",
            3,
            12,
            log=True
        ),

        auto_class_weights=trial.suggest_categorical(
            "auto_class_weights",
            [None, "SqrtBalanced"]
        ),

        random_strength=1,

        loss_function="Logloss",
        random_seed=42,
        verbose=False,
        thread_count=1
    )

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    return scores.mean()

Подбор гиперпараметров.

In [68]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=20
)

[I 2026-08-20 03:20:24,377] A new study created in memory with name: no-name-019782f6-487a-4917-adc9-377f7a7865d5
[I 2026-08-20 03:20:38,935] Trial 0 finished with value: 0.8645908550644148 and parameters: {'iterations': 600, 'learning_rate': 0.031855023524570446, 'depth': 4, 'l2_leaf_reg': 4.160533669039796, 'auto_class_weights': None}. Best is trial 0 with value: 0.8645908550644148.
[I 2026-08-20 03:20:51,240] Trial 1 finished with value: 0.8644952556777813 and parameters: {'iterations': 500, 'learning_rate': 0.03353472094400219, 'depth': 4, 'l2_leaf_reg': 11.724896126959887, 'auto_class_weights': None}. Best is trial 0 with value: 0.8645908550644148.
[I 2026-08-20 03:21:06,365] Trial 2 finished with value: 0.8656472511375597 and parameters: {'iterations': 500, 'learning_rate': 0.04734393780923698, 'depth': 6, 'l2_leaf_reg': 5.827173855859045, 'auto_class_weights': None}. Best is trial 2 with value: 0.8656472511375597.
[I 2026-08-20 03:21:24,328] Trial 3 finished with value: 0.865650

Сохранение оптимальных гиперпараметров.

In [69]:
catboost_best_params = study.best_params

# Обучение моделей с подобранными оптимальными гиперпараметрами

**Обучение логистической регрессии.**

In [70]:
log_reg = LogisticRegression(**logistic_regression_best_params)

In [71]:
log_reg.fit(X_t_train, y_t_train)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",2.085738322376854
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' i

**Обучение CatBoost классификатора.**

In [72]:
cat_classifier = CatBoostClassifier(**catboost_best_params)

In [73]:
cat_classifier.fit(X_train, y_train)

0:	learn: 0.6767574	total: 163ms	remaining: 1m 54s
1:	learn: 0.6612787	total: 176ms	remaining: 1m 1s
2:	learn: 0.6461038	total: 189ms	remaining: 44s
3:	learn: 0.6323372	total: 204ms	remaining: 35.5s
4:	learn: 0.6185627	total: 224ms	remaining: 31.1s
5:	learn: 0.6055261	total: 242ms	remaining: 28s
6:	learn: 0.5933681	total: 260ms	remaining: 25.8s
7:	learn: 0.5815611	total: 280ms	remaining: 24.2s
8:	learn: 0.5704089	total: 302ms	remaining: 23.2s
9:	learn: 0.5600939	total: 322ms	remaining: 22.2s
10:	learn: 0.5500147	total: 338ms	remaining: 21.2s
11:	learn: 0.5412253	total: 356ms	remaining: 20.4s
12:	learn: 0.5325015	total: 375ms	remaining: 19.8s
13:	learn: 0.5244363	total: 393ms	remaining: 19.3s
14:	learn: 0.5164703	total: 418ms	remaining: 19.1s
15:	learn: 0.5086135	total: 446ms	remaining: 19s
16:	learn: 0.5014019	total: 465ms	remaining: 18.7s
17:	learn: 0.4946307	total: 484ms	remaining: 18.3s
18:	learn: 0.4884391	total: 506ms	remaining: 18.1s
19:	learn: 0.4821446	total: 529ms	remaining: 1

CatBoostClassifier(auto_class_weights='SqrtBalanced', depth=7, iterations=700, l2_leaf_reg=3.019214872532525, learning_rate=0.020079631538535796)

# Оценка качества моделей

**Сводная таблица качества моделей.**

In [117]:
models_metrics = pd.DataFrame(columns=[
    "model",
    "precision",
    "recall",
    "roc_auc",
    "pr_auc",
    "gini"
])

**Вычисление метрик качества логистической регрессии.**

In [118]:
log_reg_metrics = evaluate_model(
    model=log_reg,
    X_test=X_t_test,
    y_test=y_t_test
)

In [119]:
log_reg_metrics

(0.22292292292292293,
 0.7403590425531915,
 0.857883129904683,
 0.38708975400077045,
 0.7157662598093659)

In [120]:
models_metrics.loc[ models_metrics.shape[0] ] = ("logistic regression",) + log_reg_metrics

**Вычисление метрик качества CatBoostClassifier.**

In [121]:
cat_clsfr = evaluate_model(
    model=cat_classifier,
    X_test=X_test,
    y_test=y_test
)

In [122]:
cat_clsfr

(0.41133761519128736,
 0.4896941489361702,
 0.8669059747449361,
 0.40653873966562615,
 0.7338119494898723)

In [123]:
models_metrics.loc[ models_metrics.shape[0] ] = ("catboost classifier",) + cat_clsfr

**Вычисление разницы метрик между моделями.**

In [124]:
models_metrics.loc[ models_metrics.shape[0] ] = [
    "delta",
    models_metrics.iloc[1, 1] - models_metrics.iloc[0, 1],
    models_metrics.iloc[1, 2] - models_metrics.iloc[0, 2],
    models_metrics.iloc[1, 3] - models_metrics.iloc[0, 3],
    models_metrics.iloc[1, 4] - models_metrics.iloc[0, 4],
    models_metrics.iloc[1, 5] - models_metrics.iloc[0, 5]
]

In [125]:
models_metrics

,model,precision,recall,roc_auc,pr_auc,gini
0,logistic regression,0.2229,0.7404,0.8579,0.3871,0.7158
1,catboost classifier,0.4113,0.4897,0.8669,0.4065,0.7338
2,delta,0.1884,-0.2507,0.0090,0.0194,0.0180
